# 02 — Integrated Gradient Validation: 1D

Compare DDG integrated gradient `Df_i = 0.5 * sum_j (f_j - f_i) * A_ij`
against analytically integrated gradient via the divergence theorem
`f(b) - f(a)` over the same dual cell interval.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt

from hyperct import Complex
from hyperct.ddg import compute_vd
from hyperct.ddg._dual_cell import dual_cell_vertices_1d

from ddgclib.operators.stress import scalar_gradient_integrated
from ddgclib.analytical import integrated_gradient_1d

## Helper: build 1D mesh

In [ ]:
def make_mesh_1d(n_refine=3, seed=None, jitter=0.05):
    HC = Complex(1, domain=[(0.0, 1.0)])
    HC.triangulate()
    for _ in range(n_refine):
        HC.refine_all()
    bV = set()
    for v in HC.V:
        on_bnd = abs(v.x_a[0]) < 1e-14 or abs(v.x_a[0] - 1.0) < 1e-14
        v.boundary = on_bnd
        if on_bnd: bV.add(v)
    if seed is not None:
        rng = np.random.default_rng(seed)
        for v in HC.V:
            if v not in bV and v.nn:
                el = min(np.linalg.norm(v.x_a - vn.x_a) for vn in v.nn)
                off = rng.uniform(-jitter*el, jitter*el)
                v.x = (v.x_a[0] + off,)
                v.x_a = np.array(v.x, dtype=v.dtype)
    compute_vd(HC, cdist=1e-10)
    interior = [v for v in HC.V if v not in bV]
    return HC, bV, interior

## Linear field: f(x) = 2x + 1
Expect machine precision on all meshes.

In [ ]:
f_lin = lambda x: 2.0 * x[0] + 1.0

for label, seed in [('Symmetric', None), ('Jittered', 42)]:
    HC, bV, interior = make_mesh_1d(n_refine=3, seed=seed)
    for v in HC.V:
        v.f = f_lin(v.x_a[:1])

    print(f'\n=== {label} mesh ===')
    print(f'{"x":>8}  {"Df_num":>12}  {"Df_ana":>12}  {"error":>12}')
    for v in sorted(interior, key=lambda v: v.x_a[0]):
        num = scalar_gradient_integrated(v, HC, dim=1, field_attr='f')
        a, b = dual_cell_vertices_1d(v)
        ana = integrated_gradient_1d(f_lin, a, b)
        print(f'{v.x_a[0]:>8.4f}  {num[0]:>12.8f}  {ana[0]:>12.8f}  {abs(num[0]-ana[0]):>12.2e}')

## Quadratic field: f(x) = x²
In 1D the DDG operator is exact (f(b)-f(a) matches).

In [ ]:
f_quad = lambda x: x[0]**2

HC, bV, interior = make_mesh_1d(n_refine=3, seed=42)
for v in HC.V:
    v.f = f_quad(v.x_a[:1])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot the function and dual cells
ax = axes[0]
xs = np.linspace(0, 1, 200)
ax.plot(xs, xs**2, 'k-', label='f(x) = x²')
for v in interior:
    a, b = dual_cell_vertices_1d(v)
    ax.axvspan(a, b, alpha=0.15, color='C0')
    ax.plot(v.x_a[0], v.f, 'ro', markersize=6)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('f(x) = x² with dual cell intervals')
ax.legend()

# Plot errors
ax = axes[1]
x_vals, errors = [], []
for v in sorted(interior, key=lambda v: v.x_a[0]):
    num = scalar_gradient_integrated(v, HC, dim=1, field_attr='f')
    a, b = dual_cell_vertices_1d(v)
    ana = integrated_gradient_1d(f_quad, a, b)
    x_vals.append(v.x_a[0])
    errors.append(abs(num[0] - ana[0]))

ax.semilogy(x_vals, errors, 'ro-')
ax.axhline(1e-14, color='g', linestyle='--', label='Machine precision')
ax.set_xlabel('x'); ax.set_ylabel('|error|')
ax.set_title('Error: numerical vs analytical (quadratic, jittered)')
ax.legend()

plt.tight_layout()
plt.show()
print(f'Max error: {max(errors):.2e}')

## Convergence study: f(x) = x³

In [ ]:
f_cubic = lambda x: x[0]**3

fig, ax = plt.subplots(figsize=(8, 5))

for seed, label in [(None, 'Symmetric'), (42, 'Jittered')]:
    refines = range(2, 7)
    max_errors = []
    for n_ref in refines:
        HC, bV, interior = make_mesh_1d(n_refine=n_ref, seed=seed)
        for v in HC.V:
            v.f = f_cubic(v.x_a[:1])
        errs = []
        for v in interior:
            num = scalar_gradient_integrated(v, HC, dim=1, field_attr='f')
            a, b = dual_cell_vertices_1d(v)
            ana = integrated_gradient_1d(f_cubic, a, b)
            errs.append(abs(num[0] - ana[0]))
        max_errors.append(max(errs))
    ax.semilogy(list(refines), max_errors, 'o-', label=label)

ax.set_xlabel('Refinement level')
ax.set_ylabel('Max |error|')
ax.set_title('Convergence: f(x) = x³')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()